# 04 — Revenue Gap Map

Maps the **total annual property tax revenue gap** by county — the estimated additional tax revenue
that would be collected if all residential properties were assessed at current market value
instead of their Prop 13-limited assessed values.

Formula: `Tax Gap ($) = (Market Value − Assessed Value) × 1.1%`

The 1.1% rate approximates California's 1% base rate plus average local special assessments/bonds.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'county_prop13.csv')
df['fips'] = df['fips'].astype(str).str.zfill(5)
df = df.dropna(subset=['tax_gap_annual_millions'])

total_gap = df['tax_gap_annual_millions'].sum()
print(f'Statewide annual residential property tax gap: ${total_gap:,.0f}M')
print(f'That is ${total_gap/1000:,.1f}B per year statewide')

Statewide annual residential property tax gap: $25,629M
That is $25.6B per year statewide


In [2]:
# Choropleth: total annual tax gap by county
fig = px.choropleth(
    df,
    geojson='https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json',
    locations='fips',
    color='tax_gap_annual_millions',
    color_continuous_scale='YlOrRd',
    scope='usa',
    hover_name='county',
    hover_data={
        'tax_gap_annual_millions': ':$,.0f',
        'tax_gap_per_household': ':$,.0f',
        'assessment_ratio': ':.1%',
        'fips': False,
    },
    labels={
        'tax_gap_annual_millions': 'Annual Gap ($M)',
        'tax_gap_per_household': 'Gap per HH ($)',
        'assessment_ratio': 'Assessment Ratio',
    },
    title='Annual Property Tax Revenue Gap by CA County ($M)<br>'
          '<sup>Estimated additional revenue if all homes assessed at market value</sup>',
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(
    margin={'l': 0, 'r': 0, 't': 60, 'b': 0},
    coloraxis_colorbar=dict(
        title='Annual Gap<br>($M)',
        tickprefix='$',
        ticksuffix='M',
    ),
    height=600,
)
fig.show()

In [3]:
# Choropleth: tax gap per household
fig2 = px.choropleth(
    df,
    geojson='https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json',
    locations='fips',
    color='tax_gap_per_household',
    color_continuous_scale='PuRd',
    scope='usa',
    hover_name='county',
    hover_data={
        'tax_gap_per_household': ':$,.0f',
        'assessment_ratio': ':.1%',
        'fips': False,
    },
    labels={
        'tax_gap_per_household': 'Annual Gap per HH ($)',
        'assessment_ratio': 'Assessment Ratio',
    },
    title='Annual Property Tax Gap per Owner-Occupied Household by County<br>'
          '<sup>Individual household benefit from Prop 13 assessment limits</sup>',
)
fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    margin={'l': 0, 'r': 0, 't': 60, 'b': 0},
    coloraxis_colorbar=dict(
        title='Annual Gap<br>per HH',
        tickprefix='$',
        tickformat=',.0f',
    ),
    height=600,
)
fig2.show()

In [4]:
# Summary table: top 15 counties by total gap
summary = df.nlargest(15, 'tax_gap_annual_millions')[[
    'county', 'assessment_ratio', 'market_value_millions',
    'residential_av_millions', 'tax_gap_annual_millions', 'tax_gap_per_household'
]].copy()
summary.columns = [
    'County', 'Assessment Ratio', 'Market Value ($M)',
    'Assessed Value ($M)', 'Annual Tax Gap ($M)', 'Gap per HH ($)'
]
summary.style.format({
    'Assessment Ratio': '{:.1%}',
    'Market Value ($M)': '${:,.0f}M',
    'Assessed Value ($M)': '${:,.0f}M',
    'Annual Tax Gap ($M)': '${:,.0f}M',
    'Gap per HH ($)': '${:,.0f}',
})

,County,Assessment Ratio,Market Value ($M),Assessed Value ($M),Annual Tax Gap ($M),Gap per HH ($)
18,Los Angeles,60.9%,"$1,513,787M","$922,562M","$6,503M","$4,190"
29,Orange,60.9%,"$632,198M","$384,917M","$2,720M","$4,511"
42,Santa Clara,59.1%,"$595,654M","$352,096M","$2,679M","$7,410"
36,San Diego,63.7%,"$549,144M","$349,894M","$2,192M","$3,516"
40,San Mateo,52.8%,"$292,050M","$154,305M","$1,515M","$9,628"
0,Alameda,68.4%,"$366,024M","$250,428M","$1,272M","$4,006"
37,San Francisco,58.5%,"$231,291M","$135,399M","$1,055M","$7,582"
6,Contra Costa,71.5%,"$272,062M","$194,542M",$853M,"$3,108"
35,San Bernardino,70.1%,"$194,347M","$136,253M",$639M,"$1,585"
55,Ventura,64.1%,"$144,777M","$92,758M",$572M,"$3,241"
